<a href="https://colab.research.google.com/github/AlaaSoudy/Customer_Churn_End_to_End_Project-ML-/blob/main/Customer_Churn_End_to_End_Project(ML).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# End-to-End Customer Churn Prediction Project
## Machine Learning Pipeline for Telco Customer Churn

This project builds a complete and reproducible machine learning solution to predict whether a telecom customer is likely to churn based on customer profile, account details, subscribed services, and billing information.

**Created by:** Alaa Soudy Ibrahim Ibrahim

### Project Structure
1. Data Understanding
2. Data Cleaning and Preprocessing
3. Exploratory Data Analysis and Insights
4. Feature Engineering
5. Train-Test Split
6. Model Building
7. Ensemble Learning
8. Model Evaluation
9. Conclusion
10. Streamlit Application

### Project Goals
- Understand the main customer and service factors associated with churn.
- Build reusable preprocessing and modeling pipelines.
- Compare individual classification models with ensemble methods.
- Select the final model using metrics suited to an imbalanced target.
- Package the selected pipeline inside an interactive Streamlit application.

> All random operations use `random_state=42` for reproducibility, and the test set remains untouched until final evaluation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
import warnings
warnings.filterwarnings("ignore")
from scipy.stats.contingency import association
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,roc_auc_score
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (BaggingClassifier,AdaBoostClassifier,
    GradientBoostingClassifier, VotingClassifier,StackingClassifier)
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay
)

from sklearn.metrics import RocCurveDisplay
from sklearn.base import clone


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install -q kagglehub imbalanced-learn streamlit

# 1. Data Understanding

## 1.1 Data Loading

The dataset is downloaded using `kagglehub`, then the CSV file is detected automatically so the notebook can run in both Google Colab and Kaggle-style environments.


In [ ]:
# Download and load the Telco Customer Churn dataset
path = kagglehub.dataset_download("blastchar/telco-customer-churn")
print("Dataset directory:", path)

csv_files = [f for f in os.listdir(path) if f.lower().endswith('.csv')]
if not csv_files:
    raise FileNotFoundError(f"No CSV file found in {path}")

dataset_file = os.path.join(path, csv_files[0])
df = pd.read_csv(dataset_file)
print("Loaded:", dataset_file)
print("Shape:", df.shape)


In [ ]:
df.head()

### First and Last Rows


## 1.2 Initial Data Inspection

This section checks the dataset shape, data types, descriptive statistics, unique values, missing values, and duplicate rows.


In [ ]:
display(df.tail())


In [ ]:
# 1. Shape of dataset
print("Shape of dataset:", df.shape)

# 2. Display data types
print(f"\nData types:\n{df.dtypes}")

# 3. Generate descriptive statistics
print(f"\nDescriptive statistics:\n{df.describe(include='all')}")

# 4. Check unique values
print(f"\nUnique values in data:\n{df.nunique()}")

# 5. Identify missing values
print(f"\nMissing values in data:\n{df.isnull().sum()}")

# 6. Detect duplicate records
print(f"\nDuplicate records: {df.duplicated().sum()}")

# 2. Data Cleaning and Preprocessing

`customerID` is removed because it is an identifier rather than a predictive feature. `TotalCharges` is converted to numeric and missing values are handled.


### Cleaning Decisions and Justification
- Strip whitespace from column names and categorical values to prevent inconsistent duplicate categories.
- Convert `TotalCharges` to numeric because blank strings are stored as text.
- Remove exact duplicate records so repeated observations do not bias training.
- Remove `customerID` because it is a unique identifier, not a reusable predictor.
- Treat `SeniorCitizen` as categorical.
- Diagnose outliers using IQR and box plots. Valid charge and tenure values are retained because they represent real customers.


In [ ]:
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()

rows_before = len(df)
df.drop_duplicates(inplace=True)
print('Duplicate rows removed:', rows_before - len(df))


In [ ]:
df.drop("customerID", axis=1, inplace=True)

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(df['TotalCharges'].dtype)
print(df['TotalCharges'].isnull().sum())
df[df['TotalCharges'].isnull()]

In [ ]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)
print(df.isnull().sum())
print(df['TotalCharges'].dtype)

In [ ]:
df['SeniorCitizen'] = df['SeniorCitizen'].astype('category')
df["SeniorCitizen"].dtypes
df["SeniorCitizen"].isnull()

In [ ]:
def Identifynumandcatfeatures():
    numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()
    return numerical_features, categorical_features
Identifynumandcatfeatures()

**Dataset Description:**


The Telco Customer Churn Dataset contains information about telecom customers, including demographic details, subscribed services, contract types, payment methods, and monthly and total charges.


Target Variable (Churn):


The target variable is Churn, which indicates whether a customer has left the company (Yes) or is still subscribed (No).


This is a Binary Classification Problem because the model predicts one of two possible outcomes.

# 3. Exploratory Data Analysis and Insights

The analysis examines outliers, numerical correlations, categorical associations, distributions, and relationships with the churn target.


In [ ]:
def detect_outliers(df, numerical_features):
    for col in numerical_features:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outliers = df[(df[col] < lower) | (df[col] > upper)]

        print(f'{col}: {len(outliers)} Outliers')
numerical_features, categorical_features = Identifynumandcatfeatures()
detect_outliers(df, numerical_features)

In [ ]:
for col in numerical_features:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=df, x=col)
    plt.title(f'Boxplot of {col}')
    plt.show()

In [ ]:
numerical_features

In [ ]:
df_corr = df.copy()
df_corr['Churn'] = df_corr['Churn'].map({'No': 0, 'Yes': 1})
sns.heatmap(
    df_corr[numerical_features + ['Churn']].corr(),
    annot=True,
    cmap='coolwarm'
)
plt.show()

The correlation heatmap illustrates the relationships between numerical features.

TotalCharges has a strong positive correlation with Tenure because customers who stay longer generally accumulate higher total charges.

MonthlyCharges also contributes to TotalCharges, since the total amount depends partly on the monthly payment and subscription duration.

These relationships help identify possible dependencies between numerical features and provide useful information before building the prediction model.

In [ ]:
cramers_matrix = pd.DataFrame(
    index=categorical_features,
    columns=categorical_features,
    dtype=float
)

for col1 in categorical_features:
    for col2 in categorical_features:

        table = pd.crosstab(df[col1], df[col2])

        cramers_matrix.loc[col1, col2] = association(
            table,
            method='cramer'
        )
plt.figure(figsize=(16, 12))

sns.heatmap(
    cramers_matrix,
    annot=True,
    cmap='coolwarm',
    vmin=0,
    vmax=1,
    fmt='.2f'
)

plt.title("Cramér's V Heatmap")
plt.show()

The Cramér's V heatmap measures the strength of association between categorical features.

It helps identify which categorical variables have stronger relationships with each other and with the target variable Churn.

Features with stronger associations with Churn may be useful for prediction, while highly associated input features may contain overlapping information.

In [ ]:
for col in numerical_features:
    plt.figure(figsize=(7, 4))
    sns.histplot(data=df, x=col, kde=True)
    plt.title(f'Distribution of {col}')
    plt.show()

The histograms show the distributions of Tenure, MonthlyCharges, and TotalCharges.

- Tenure has a bimodal distribution, indicating groups of newly joined and long-term customers.
- MonthlyCharges shows different concentration areas, possibly reflecting different service packages.
- TotalCharges is strongly right-skewed, meaning most customers have relatively low total charges.

These patterns can help identify customer groups and understand how subscription duration and payment amounts may relate to churn.

In [ ]:
for col in categorical_features:
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x=col, hue=col, palette='Set2', legend=False)
    plt.title(f'Distribution of {col}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


The charts compare churn rates across different categorical customer groups.

- Month-to-month customers have a much higher churn rate than long-term contract customers.
- Electronic check users show the highest churn rate among payment methods.
- Fiber optic customers have a higher churn rate than DSL customers.
- Customers without OnlineSecurity and TechSupport tend to have higher churn rates.
- Senior citizens also show higher churn rates than non-senior customers.

These findings help identify customer segments that may be at higher risk of leaving the company.

In [ ]:
plt.figure(figsize=(6, 5))
sns.barplot(data=df, x='Churn', y='tenure')
plt.show()

**Observation:** This chart compares average tenure by churn status. Customers who churn generally have shorter tenure, making tenure a useful retention-risk signal.


In [ ]:
plt.figure(figsize=(6, 5))
plt.pie(df['Churn'].value_counts(), labels=['No', 'Yes'], autopct='%1.1f%%')
plt.title('Churn Distribution')
plt.show()

**Observation:** The pie chart shows that churners are the minority class. Therefore, Accuracy must be reported with Precision, Recall, F1-score, and ROC AUC.


In [ ]:
sns.pairplot(
    data=df,
    vars=numerical_features,
    hue='Churn'
)
plt.show()

**Observation:** The pairplot shows substantial overlap between churn classes, so no single numerical feature perfectly separates them. Comparing linear, distance-based, tree-based, and ensemble models is justified.


In [ ]:
sns.countplot(data=df, x='Churn')
plt.title('Churn Distribution')
plt.show()

print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True) * 100)

The chart represents the distribution of customers who churned and those who stayed.

Most customers (73.46%) stayed with the company, while 26.54% churned. This indicates a noticeable class imbalance.

This finding is important for prediction because accuracy alone may not be sufficient to evaluate the model. Metrics such as Precision, Recall, and F1-score should also be considered.

In [ ]:
for col in categorical_features:
    if col != 'Churn':
        print(f'\n{col}:')
        print(df[col].value_counts())
        print(df[col].value_counts(normalize=True) * 100)

In [ ]:
for col in categorical_features:
    if col != 'Churn':
        churn_rate = pd.crosstab(
            df[col],
            df['Churn'],
            normalize='index'
        ) * 100

        churn_rate.plot(
            kind='bar',
            stacked=True,
            figsize=(8, 4)
        )

        plt.title(f'{col} vs Churn')
        plt.ylabel('Percentage (%)')
        plt.xticks(rotation=45)
        plt.legend(title='Churn')
        plt.show()

In [ ]:
for col in numerical_features:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=df, x='Churn', y=col)
    plt.title(f'{col} vs Churn')
    plt.show()

In [ ]:
plt.figure(figsize=(20, 25))

for i, col in enumerate(categorical_features):
    plt.subplot(6, 3, i + 1)

    sns.countplot(data=df, x=col, hue=col, palette='Set2', legend=False)

    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

The charts represent the frequency distribution of categorical customer features.

- Gender is almost equally distributed between males and females.
- Most customers are not senior citizens.
- Month-to-month contracts represent the largest contract group (55.02%).
- Electronic check is the most common payment method (33.58%).
- A considerable number of customers do not subscribe to additional services such as OnlineSecurity and TechSupport.

These distributions help identify common customer characteristics and provide an initial understanding of the customer base. They also help prepare categorical features for churn prediction.

In [ ]:
for col in categorical_features:
    if col != 'Churn':
        print(f'\n{col} vs Churn:')

        display(
            pd.crosstab(
                df[col],
                df['Churn'],
                normalize='index'
            ) * 100
        )

In [ ]:
for col in numerical_features:
    print(f'\n{col} vs Churn:')

    groups = pd.qcut(df[col], q=5, duplicates='drop')

    display(pd.crosstab(
        groups,
        df['Churn'],
        normalize='index'
    ) * 100)

### Churn Distribution


The dataset contains 7,043 customers. Approximately 73.46% of customers stayed with the company, while 26.54% discontinued their services.


This indicates a noticeable class imbalance, as the number of customers who stayed is significantly higher than the number who churned. Therefore, evaluation metrics such as Recall, Precision, and F1-score should be considered alongside Accuracy when evaluating the prediction model.

### Numerical Features Distribution

**1. Tenure:**

The distribution is bimodal, with two distinct peaks: one at the lowest end (0–5 months) and another at the highest end (around 70–72 months). This indicates two major customer groups: newly joined customers and long-term customers.

**2. MonthlyCharges:**

The distribution shows two main concentration areas: a sharp peak around 20 USD and a wider peak between 70 USD and 105 USD. This may indicate differences in customer service packages and subscriptions.

**3. TotalCharges:**

The distribution is strongly right-skewed, with most customers concentrated at lower total charges and fewer customers at higher values. This is likely related to differences in subscription duration and monthly charges.

### Categorical Features Distribution

**1. Demographic Features:**

The gender distribution is nearly balanced, with males representing 50.48% and females 49.52% of customers. Most customers are not senior citizens (83.79%), and approximately 70% do not have dependents.

**2. Service Features:**

PhoneService is the most common service, used by 90.32% of customers. Fiber optic is the most popular internet service (43.96%). A considerable proportion of customers do not subscribe to additional services such as OnlineSecurity and TechSupport. StreamingTV and StreamingMovies have relatively similar distributions between Yes and No.

**3. Contract and Payment Features:**

Month-to-month contracts represent the largest group (55.02%), followed by two-year and one-year contracts. Most customers use PaperlessBilling (59.22%). Electronic check is the most common payment method (33.58%).

### Relationship Between Important Features and the Target Variable

**1. Numerical Features:**

- **Tenure:** Shows a strong negative relationship with churn. Customers with shorter subscription periods are more likely to leave the company.
- **MonthlyCharges:** Generally shows a positive relationship with churn. Customers paying higher monthly charges tend to have higher churn rates.
- **TotalCharges:** Generally shows a negative relationship with churn. Customers with lower total charges tend to have higher churn rates, which may be related to shorter tenure.

**2. Categorical Features:**

- **Contract:** Month-to-month customers have a much higher churn rate than customers with one-year or two-year contracts.
- **PaymentMethod:** Electronic check users show the highest churn rate compared to other payment methods.
- **InternetService:** Fiber optic customers have a higher churn rate than DSL customers.
- **TechSupport and OnlineSecurity:** Customers without these services tend to have higher churn rates.
- **SeniorCitizen:** Senior citizens show a higher churn rate than non-senior customers.
- **Partner and Dependents:** Customers without partners or dependents tend to have higher churn rates.

**Conclusion:**

Contract, tenure, PaymentMethod, InternetService, and support-related services show noticeable relationships with customer churn. These features may be important for predicting customer churn.

### Customer Groups More Likely to Churn

Based on the exploratory data analysis, several customer groups show higher churn rates:

1. **New Customers:** Customers with 0–6 months of tenure have the highest churn rate (52.94%).

2. **Month-to-Month Customers:** Customers with month-to-month contracts have a high churn rate (42.71%), compared to only 2.83% for two-year contract customers.

3. **Electronic Check Users:** Customers using electronic checks have the highest churn rate among payment methods (45.29%).

4. **Fiber Optic Users:** Customers using fiber optic internet have a higher churn rate (41.89%) compared to DSL users (18.96%).

5. **Customers Without Additional Services:** Customers without OnlineSecurity and TechSupport have higher churn rates (41.77% and 41.64%, respectively).

6. **Senior Citizens:** Senior citizens have a higher churn rate (41.68%) compared to non-senior customers (23.61%).

**Conclusion:**

New customers, month-to-month contract customers, electronic check users, and customers without additional support services are among the groups more likely to churn. These groups could be important targets for customer retention strategies.

### Services Associated with Lower Customer Churn

Based on the exploratory data analysis, some additional services are associated with lower customer churn rates:

1. **OnlineSecurity:** Customers with this service have a lower churn rate (14.61%) compared to customers without it (41.77%).

2. **TechSupport:** Customers with technical support have a lower churn rate (15.17%) compared to those without it (41.64%).

3. **OnlineBackup:** Customers with online backup have a lower churn rate (21.53%) compared to those without it (39.93%).

4. **DeviceProtection:** Customers with device protection have a lower churn rate (22.50%) compared to those without it (39.13%).

**Conclusion:**

OnlineSecurity and TechSupport show the strongest associations with lower customer churn. These services may contribute to customer retention and could be considered in customer retention strategies.

Note: These results show associations, not proof that subscribing to these services directly causes lower churn.

### Features with the Strongest Relationship with Churn

Based on the exploratory data analysis, the following features show the strongest relationships with customer churn:

1. **Tenure:** Customers with shorter subscription periods are much more likely to churn than long-term customers.

2. **Contract:** Month-to-month customers have a significantly higher churn rate than customers with long-term contracts.

3. **PaymentMethod:** Customers using electronic checks show the highest churn rate among payment methods.

4. **InternetService:** Fiber optic customers have a higher churn rate compared to DSL customers.

5. **OnlineSecurity and TechSupport:** Customers without these services show considerably higher churn rates.

6. **MonthlyCharges:** Higher monthly charges are generally associated with higher churn rates.

**Conclusion:**

Tenure, Contract, PaymentMethod, InternetService, OnlineSecurity, and TechSupport appear to be among the most important features associated with customer churn.

These findings are based on exploratory analysis and do not establish causation or a definitive feature-importance ranking. Further statistical analysis and machine learning models are needed to measure their predictive importance.

### Noticeable Trends and Interesting Patterns

Several interesting patterns were observed during the exploratory data analysis:

1. **Tenure:** New customers are more likely to churn. The churn rate is 52.94% for customers with 0–6 months of tenure, compared to only 6.61% for customers with 60–72 months.

2. **Contract Type:** Month-to-month customers have a much higher churn rate (42.71%) than two-year contract customers (2.83%).

3. **Additional Services:** Customers subscribed to OnlineSecurity and TechSupport show considerably lower churn rates than customers without these services.

4. **Payment Method:** Electronic check users have the highest churn rate (45.29%) among all payment methods.

5. **Monthly Charges:** Higher monthly charges are generally associated with higher churn rates, although this trend is not consistent across all charge groups.

### Unusual Observations

1. **High Churn Among New Customers:** More than half of customers with 0–6 months of tenure leave the company.

2. **Very Low Churn for Two-Year Contracts:** Only 2.83% of customers with two-year contracts churn.

3. **Fiber Optic Churn:** Fiber optic customers have a higher churn rate (41.89%) than DSL customers (18.96%), which is an interesting and unexpected pattern.

4. **TotalCharges Distribution:** TotalCharges is strongly right-skewed, with most customers having relatively low total charges and a smaller number having very high values.

5. **Gender:** Male and female customers have nearly identical churn rates, indicating little difference between the two groups.

**Overall Conclusion:**

Customer tenure and contract type show particularly noticeable patterns. New customers and month-to-month subscribers are more likely to churn, while long-term customers and customers subscribed to additional services tend to have lower churn rates.

# Final EDA Summary

After analyzing the Telco Customer Churn dataset, the following insights were identified:

1. **Churn Distribution:** 26.54% of customers left the company, while 73.46% stayed.
2. **Tenure:** New customers (0–6 months) have a higher churn rate (52.94%) than long-term customers (6.61%).
3. **Contract:** Month-to-month customers show higher churn (42.71%) than two-year contract customers (2.83%).
4. **Payment Method:** Electronic check users have the highest churn rate (45.29%).
5. **Internet Service:** Fiber optic users have higher churn (41.89%) than DSL users (18.96%).
6. **Additional Services:** Customers without OnlineSecurity and TechSupport are more likely to churn.
7. **Monthly Charges:** Higher monthly charges are generally associated with higher churn.

### Final Conclusion

Tenure, Contract, PaymentMethod, InternetService, OnlineSecurity, and TechSupport are important features associated with customer churn. These insights can help identify high-risk customers and build a machine learning model to predict churn.

**Note:** These findings show associations, not proven causes.

# 4. Feature Engineering

New features summarize customer spending, subscribed services, protection services, and tenure groups. The target is encoded as `0 = No Churn` and `1 = Churn`.


In [ ]:
X = df.drop('Churn', axis=1).copy()
y = df['Churn'].map({'No': 0, 'Yes': 1})

# Binary Mapping
binary_mapping = {
    'Yes': 1,
    'No': 0,
    'Male': 1,
    'Female': 0
}

binary_columns = [
    'gender',
    'Partner',
    'Dependents',
    'PhoneService',
    'PaperlessBilling'
]

for col in binary_columns:
    X[col] = X[col].map(binary_mapping)


# Feature Engineering

# 1. Average Monthly Spend
X['AvgMonthlySpend'] = np.where(
    X['tenure'] > 0,
    X['TotalCharges'] / X['tenure'],
    X['MonthlyCharges']
)

# 2. Number of Services
service_columns = [
    'PhoneService',
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies'
]

X['NumServices'] = (
    (X['PhoneService'] == 1).astype(int)
    + (X['MultipleLines'] == 'Yes').astype(int)
    + (X['InternetService'] != 'No').astype(int)
    + (X[['OnlineSecurity', 'OnlineBackup',
          'DeviceProtection', 'TechSupport',
          'StreamingTV', 'StreamingMovies']] == 'Yes').sum(axis=1)
)

# 3. Number of Protection Services
protection_columns = [
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport'
]

X['NumProtectionServices'] = (
    X[protection_columns] == 'Yes'
).sum(axis=1)

# 4. Tenure Group
X['TenureGroup'] = pd.cut(
    X['tenure'],
    bins=[-1, 12, 24, 48, float('inf')],
    labels=['New', 'Short', 'Medium', 'Long']
)

In [ ]:
numerical_features = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges',
    'AvgMonthlySpend',
    'NumServices',
    'NumProtectionServices'
]

In [ ]:
if 'TenureGroup' not in categorical_features:
    categorical_features.append('TenureGroup')


In [ ]:
X[['AvgMonthlySpend', 'NumServices',
   'NumProtectionServices', 'TenureGroup']].head(10)

In [ ]:
for col in [
    'AvgMonthlySpend',
    'NumServices',
    'NumProtectionServices',
    'TenureGroup'
]:
    print(f'\n{col}')
    print(df.assign(
        Churn=df['Churn']
    ).groupby(
        X[col], observed=True
    )['Churn'].value_counts(normalize=True))

In [ ]:
spend_groups = pd.qcut(
    X['AvgMonthlySpend'],
    q=5,
    duplicates='drop'
)

pd.crosstab(
    spend_groups,
    y,
    normalize='index'
) * 100

In [ ]:
X[['AvgMonthlySpend', 'MonthlyCharges']].corr()

In [ ]:
X.drop('AvgMonthlySpend', axis=1, inplace=True)

numerical_features.remove('AvgMonthlySpend')

In [ ]:
numerical_features = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges',
    'NumServices',
    'NumProtectionServices'
]

In [ ]:
print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)

print("\nX Shape:", X.shape)
print("\nMissing Values:")
print(X.isnull().sum().sum())

In [ ]:
categorical_features = [
    'gender',
    'SeniorCitizen',
    'Partner',
    'Dependents',
    'PhoneService',
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaperlessBilling',
    'PaymentMethod',
    'TenureGroup'
]

In [ ]:
print(categorical_features)
print('Churn' in categorical_features)
print(categorical_features.count('TenureGroup'))

# 5. Train-Test Split

A stratified 80/20 split preserves the original churn ratio in both training and test sets.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

## Encoding and Scaling Pipeline

Numerical features are standardized and multi-class categorical features are one-hot encoded. Binary features are passed through after mapping.


In [ ]:
onehot_columns = [
    col for col in categorical_features
    if col not in binary_columns and col != 'SeniorCitizen'
]
def create_preprocessor():
    return ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            ), onehot_columns)
        ],
        remainder='passthrough'
    )

In [ ]:
categorical_columns = [
    col for col in categorical_features
    if col in X_train.columns
]

# 6. Model Building

The project compares six individual models: Logistic Regression, Decision Tree, Random Forest, KNN, SVM, and Naive Bayes. Five-fold stratified cross-validation compares them fairly.

# 7. Ensemble Learning

Bagging, AdaBoost, Gradient Boosting, Voting, and the Stacking Classifier are compared with the individual models.

Multiple classifiers are compared using 5-fold stratified cross-validation. Because the target is imbalanced, the comparison includes Precision, Recall, F1-score, and ROC AUC in addition to Accuracy.


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),

    'Decision Tree': DecisionTreeClassifier(random_state=42),

    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    'KNN': KNeighborsClassifier(n_neighbors=5),

    'SVM': SVC(probability=True, random_state=42),

    'Naive Bayes': GaussianNB()
}
ensemble_models = {

    'Bagging': BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    'AdaBoost': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=100,
        learning_rate=0.5,
        random_state=42
    ),

    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    ),

    'Voting': VotingClassifier(
        estimators=[
            ('lr', LogisticRegression(max_iter=1000)),
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('nb', GaussianNB())
        ],
        voting='soft'
    ),

    'Stacking': StackingClassifier(
        estimators=[
            ('lr', LogisticRegression(max_iter=1000)),
            ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('nb', GaussianNB())
        ],
        final_estimator=LogisticRegression(max_iter=1000),
        cv=5,
        n_jobs=-1
    )
}

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

cv_results = []

for name, model in models.items():

    model_pipeline = Pipeline(steps=[
        ('preprocessor', create_preprocessor()),
        ('model', model)
    ])

    scores = cross_validate(
        model_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({
        'Model': name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1 Score': scores['test_f1'].mean(),
        'ROC AUC': scores['test_roc_auc'].mean()
    })

cv_results_df = pd.DataFrame(cv_results)

cv_results_df = cv_results_df.sort_values(
    by='F1 Score',
    ascending=False
)

print(cv_results_df)

In [ ]:
ensemble_cv_results = []

for name, model in ensemble_models.items():

    model_pipeline = Pipeline(steps=[
        ('preprocessor', create_preprocessor()),
        ('model', model)
    ])

    scores = cross_validate(
        model_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    ensemble_cv_results.append({
        'Model': name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1 Score': scores['test_f1'].mean(),
        'ROC AUC': scores['test_roc_auc'].mean()
    })

ensemble_cv_results_df = pd.DataFrame(ensemble_cv_results)

In [ ]:
all_cv_results_df = pd.concat(
    [cv_results_df, ensemble_cv_results_df],
    ignore_index=True
)

all_cv_results_df = all_cv_results_df.sort_values(
    by='F1 Score',
    ascending=False
)

print(all_cv_results_df)

## 9. Handling Class Imbalance

Class weights and SMOTENC are tested to improve minority-class detection. Oversampling is performed inside cross-validation to avoid data leakage.


In [ ]:
from sklearn.base import clone

balanced_models = {
    'Logistic Regression Balanced': LogisticRegression(
        max_iter=1000,
        class_weight='balanced'
    ),

    'Random Forest Balanced': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced'
    )
}

balanced_results = []

for name, model in balanced_models.items():

    pipeline = Pipeline(steps=[
        ('preprocessor', create_preprocessor()),
        ('model', model)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    balanced_results.append({
        'Model': name,
        'Accuracy': scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1 Score': scores['test_f1'].mean(),
        'ROC AUC': scores['test_roc_auc'].mean()
    })

balanced_results_df = pd.DataFrame(balanced_results)

print(balanced_results_df)

In [ ]:
categorical_columns = [
    col for col in categorical_features
    if col in X_train.columns
]

cat_indices = [
    X_train.columns.get_loc(col)
    for col in categorical_columns
]

print(categorical_columns)
print(cat_indices)

In [ ]:
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline

smote_model = ImbPipeline(steps=[
    ('smote', SMOTENC(
        categorical_features=cat_indices,
        random_state=42
    )),
    ('preprocessor', create_preprocessor()),
    ('model', LogisticRegression(max_iter=1000))
])

smote_scores = cross_validate(
    smote_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

smote_results = {
    'Model': 'Logistic Regression + SMOTENC',
    'Accuracy': smote_scores['test_accuracy'].mean(),
    'Precision': smote_scores['test_precision'].mean(),
    'Recall': smote_scores['test_recall'].mean(),
    'F1 Score': smote_scores['test_f1'].mean(),
    'ROC AUC': smote_scores['test_roc_auc'].mean()
}

print(pd.DataFrame([smote_results]))

## 10. Hyperparameter Tuning

Grid search tunes Logistic Regression, Gradient Boosting, and Voting Classifier settings using F1-score as the selection metric.


In [ ]:
from sklearn.model_selection import GridSearchCV

lr_pipeline = Pipeline(steps=[
    ('preprocessor', create_preprocessor()),
    ('model', LogisticRegression(max_iter=2000))
])

param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__class_weight': [None, 'balanced']
}

grid_search = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV F1 Score:", grid_search.best_score_)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

gb_pipeline = Pipeline(steps=[
    ('preprocessor', create_preprocessor()),
    ('model', GradientBoostingClassifier(random_state=42))
])

gb_param_grid = {
    'model__n_estimators': [50, 100, 150, 200],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__max_depth': [2, 3, 4],
    'model__subsample': [0.8, 1.0]
}

gb_grid = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)

gb_grid.fit(X_train, y_train)

print("Best Parameters:", gb_grid.best_params_)
print("Best CV F1:", gb_grid.best_score_)

In [ ]:
best_gb_model = gb_grid.best_estimator_

y_pred_gb = best_gb_model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred_gb))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_gb))

print("Test F1:",
      f1_score(y_test, y_pred_gb))

print("Test ROC AUC:",
      roc_auc_score(
          y_test,
          best_gb_model.predict_proba(X_test)[:, 1]
      ))

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score
)

best_gb_model = gb_grid.best_estimator_

y_pred_gb = best_gb_model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred_gb))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_gb))

print("Test F1:", f1_score(y_test, y_pred_gb))

print("Test ROC AUC:",
      roc_auc_score(
          y_test,
          best_gb_model.predict_proba(X_test)[:, 1]
      ))

In [ ]:
voting_pipeline = Pipeline(steps=[
    ('preprocessor', create_preprocessor()),
    ('model', VotingClassifier(
        estimators=[
            ('lr', LogisticRegression(
                max_iter=1000,
                class_weight='balanced'
            )),
            ('rf', RandomForestClassifier(
                n_estimators=100,
                random_state=42
            )),
            ('nb', GaussianNB())
        ],
        voting='soft'
    ))
])

voting_param_grid = {
    'model__weights': [
        [1, 1, 1],
        [2, 1, 1],
        [3, 1, 1],
        [1, 1, 2],
        [2, 1, 2],
        [3, 1, 2]
    ]
}

voting_grid = GridSearchCV(
    estimator=voting_pipeline,
    param_grid=voting_param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)

voting_grid.fit(X_train, y_train)

print("Best Parameters:", voting_grid.best_params_)
print("Best CV F1:", voting_grid.best_score_)

In [ ]:
# 1. Logistic Regression Balanced
lr_best = Pipeline(steps=[
    ('preprocessor', create_preprocessor()),
    ('model', LogisticRegression(
        C=1,
        class_weight='balanced',
        max_iter=2000
    ))
])


# 2. Voting Tuned
voting_best = Pipeline(steps=[
    ('preprocessor', create_preprocessor()),
    ('model', VotingClassifier(
        estimators=[
            ('lr', LogisticRegression(
                max_iter=1000,
                class_weight='balanced'
            )),
            ('rf', RandomForestClassifier(
                n_estimators=100,
                random_state=42
            )),
            ('nb', GaussianNB())
        ],
        voting='soft',
        weights=[3, 1, 1]
    ))
])


# 3. Logistic Regression + SMOTENC
smote_best = ImbPipeline(steps=[
    ('smote', SMOTENC(
        categorical_features=cat_indices,
        random_state=42
    )),
    ('preprocessor', create_preprocessor()),
    ('model', LogisticRegression(
        max_iter=1000
    ))
])

# 8. Model Evaluation

All six individual models and five ensemble models are fitted on the same training set and evaluated once on the untouched test set. The comparison includes Accuracy, Precision, Recall, F1-score, ROC AUC, confusion matrices, classification reports, and ROC curves.


In [ ]:
# Combine all individual and ensemble models defined above.
all_models = {**models, **ensemble_models}
fitted_models = {}
test_results = []

for name, estimator in all_models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', create_preprocessor()),
        ('model', clone(estimator))
    ])
    pipeline.fit(X_train, y_train)
    fitted_models[name] = pipeline

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]
    test_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1 Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC AUC': roc_auc_score(y_test, y_prob)
    })

comparison_df = pd.DataFrame(test_results).sort_values(
    ['F1 Score', 'ROC AUC'], ascending=False
).reset_index(drop=True)
display(comparison_df.style.format({
    'Accuracy':'{:.3f}', 'Precision':'{:.3f}', 'Recall':'{:.3f}',
    'F1 Score':'{:.3f}', 'ROC AUC':'{:.3f}'
}))

best_model_name = comparison_df.loc[0, 'Model']
final_model = fitted_models[best_model_name]
print('Best-performing model based on F1-score:', best_model_name)


### Comparison Chart for All Trained Models


In [ ]:
plot_df = comparison_df.melt(
    id_vars='Model',
    value_vars=['Accuracy','Precision','Recall','F1 Score','ROC AUC'],
    var_name='Metric', value_name='Score'
)
plt.figure(figsize=(16, 7))
sns.barplot(data=plot_df, x='Model', y='Score', hue='Metric')
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right')
plt.title('Individual and Ensemble Model Comparison')
plt.tight_layout()
plt.show()


**Observation:** The chart compares every model using the same test set. The final model is selected by F1-score because churn is imbalanced; ROC AUC is used as a tie-breaker. This avoids selecting a model only because it predicts the majority class well.


### Classification Reports and Confusion Matrices


In [ ]:
n_models = len(fitted_models)
fig, axes = plt.subplots(4, 3, figsize=(17, 20))
axes = axes.ravel()

for ax, (name, pipeline) in zip(axes, fitted_models.items()):
    y_pred = pipeline.predict(X_test)
    print('' + '=' * 72)
    print(name)
    print(classification_report(
        y_test, y_pred, target_names=['No Churn', 'Churn'], zero_division=0
    ))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

for ax in axes[n_models:]:
    ax.axis('off')
plt.tight_layout()
plt.show()


**Observation:** False negatives are churners the model fails to identify, so they are especially costly for a retention campaign. The confusion matrices make this error visible, while the classification reports show churn-class Precision, Recall, and F1-score.


### ROC Curves


In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
for name, pipeline in fitted_models.items():
    RocCurveDisplay.from_estimator(pipeline, X_test, y_test, name=name, ax=ax)
ax.plot([0,1], [0,1], 'k--', label='Random')
ax.set_title('ROC Curves for All Models')
plt.tight_layout()
plt.show()


**Observation:** A curve closer to the upper-left corner provides better discrimination across thresholds. ROC AUC complements F1-score, which describes performance at the default classification threshold.


# 9. Conclusion


In [ ]:
best = comparison_df.iloc[0]
print('Best model:', best['Model'])
print(f"Accuracy:  {best['Accuracy']:.3f}")
print(f"Precision: {best['Precision']:.3f}")
print(f"Recall:    {best['Recall']:.3f}")
print(f"F1-score:  {best['F1 Score']:.3f}")
print(f"ROC AUC:   {best['ROC AUC']:.3f}")


### Final Discussion
- **Preprocessing:** `TotalCharges` was converted to numeric, duplicates and the ID were removed, categorical variables were encoded, and numerical features were scaled inside a pipeline to avoid leakage.
- **EDA:** Important risk signals include short tenure, month-to-month contracts, electronic-check payment, fiber optic service, higher monthly charges, and the absence of Online Security or Tech Support.
- **Best model:** The code above identifies the highest-F1 model after execution and reports its complete metric profile.
- **Selection reason:** F1-score balances Precision and Recall for the minority churn class; ROC AUC confirms ranking quality.
- **Future work:** Tune the decision threshold using campaign costs, perform nested hyperparameter tuning, add SHAP or permutation importance, calibrate probabilities, and validate on a future time period before deployment.


# 10. Streamlit Application


In [ ]:
import joblib
joblib.dump(final_model, 'churn_model.pkl')
print('Saved best complete pipeline as churn_model.pkl')


In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import joblib

# Load trained model
model = joblib.load('churn_model.pkl')

st.set_page_config(
    page_title="Customer Churn Prediction",
    page_icon="📊",
    layout="wide"
)

st.title("📊 Customer Churn Prediction")
st.write("Predict whether a customer is likely to leave the service.")

st.divider()

with st.form("customer_form"):

    st.subheader("Personal Information")

    col1, col2 = st.columns(2)

    with col1:
        gender = st.selectbox("Gender", ["Male", "Female"])
        senior_citizen = st.selectbox("Senior Citizen", [0, 1])
        partner = st.selectbox("Partner", ["Yes", "No"])
        dependents = st.selectbox("Dependents", ["Yes", "No"])

    with col2:
        tenure = st.number_input(
            "Tenure (months)",
            min_value=0,
            max_value=100,
            value=12
        )

        phone_service = st.selectbox(
            "Phone Service",
            ["Yes", "No"]
        )

        if phone_service == "Yes":
            multiple_lines = st.selectbox(
                "Multiple Lines",
                ["Yes", "No"]
            )
        else:
            multiple_lines = "No phone service"
            st.info("Multiple Lines: No phone service")

    st.divider()

    st.subheader("Internet Services")

    internet_service = st.selectbox(
        "Internet Service",
        ["DSL", "Fiber optic", "No"]
    )

    if internet_service != "No":

        col1, col2 = st.columns(2)

        with col1:
            online_security = st.selectbox(
                "Online Security", ["Yes", "No"]
            )
            online_backup = st.selectbox(
                "Online Backup", ["Yes", "No"]
            )
            device_protection = st.selectbox(
                "Device Protection", ["Yes", "No"]
            )

        with col2:
            tech_support = st.selectbox(
                "Tech Support", ["Yes", "No"]
            )
            streaming_tv = st.selectbox(
                "Streaming TV", ["Yes", "No"]
            )
            streaming_movies = st.selectbox(
                "Streaming Movies", ["Yes", "No"]
            )

    else:
        online_security = "No internet service"
        online_backup = "No internet service"
        device_protection = "No internet service"
        tech_support = "No internet service"
        streaming_tv = "No internet service"
        streaming_movies = "No internet service"

        st.info("Internet add-on services: No internet service")

    st.divider()

    st.subheader("Account Information")

    col1, col2 = st.columns(2)

    with col1:
        contract = st.selectbox(
            "Contract",
            ["Month-to-month", "One year", "Two year"]
        )

        paperless_billing = st.selectbox(
            "Paperless Billing",
            ["Yes", "No"]
        )

    with col2:
        payment_method = st.selectbox(
            "Payment Method",
            [
                "Electronic check",
                "Mailed check",
                "Bank transfer (automatic)",
                "Credit card (automatic)"
            ]
        )

    monthly_charges = st.number_input(
        "Monthly Charges",
        min_value=0.0,
        value=70.0
    )

    total_charges = st.number_input(
        "Total Charges",
        min_value=0.0,
        value=840.0
    )

    submitted = st.form_submit_button(
        "Predict Churn",
        use_container_width=True
    )


if submitted:

    # Binary Encoding
    binary_mapping = {
        "Yes": 1,
        "No": 0,
        "Male": 1,
        "Female": 0
    }

    # Feature Engineering

    num_services = (
        int(phone_service == "Yes")
        + int(multiple_lines == "Yes")
        + int(internet_service != "No")
        + sum([
            int(online_security == "Yes"),
            int(online_backup == "Yes"),
            int(device_protection == "Yes"),
            int(tech_support == "Yes"),
            int(streaming_tv == "Yes"),
            int(streaming_movies == "Yes")
        ])
    )

    num_protection_services = sum([
        int(online_security == "Yes"),
        int(online_backup == "Yes"),
        int(device_protection == "Yes"),
        int(tech_support == "Yes")
    ])

    if tenure <= 12:
        tenure_group = "New"
    elif tenure <= 24:
        tenure_group = "Short"
    elif tenure <= 48:
        tenure_group = "Medium"
    else:
        tenure_group = "Long"

    # Prepare input data
    input_data = pd.DataFrame([{
        'gender': binary_mapping[gender],
        'SeniorCitizen': senior_citizen,
        'Partner': binary_mapping[partner],
        'Dependents': binary_mapping[dependents],
        'tenure': tenure,
        'PhoneService': binary_mapping[phone_service],
        'MultipleLines': multiple_lines,
        'InternetService': internet_service,
        'OnlineSecurity': online_security,
        'OnlineBackup': online_backup,
        'DeviceProtection': device_protection,
        'TechSupport': tech_support,
        'StreamingTV': streaming_tv,
        'StreamingMovies': streaming_movies,
        'Contract': contract,
        'PaperlessBilling': binary_mapping[paperless_billing],
        'PaymentMethod': payment_method,
        'MonthlyCharges': monthly_charges,
        'TotalCharges': total_charges,
        'NumServices': num_services,
        'NumProtectionServices': num_protection_services,
        'TenureGroup': tenure_group
    }])

    # Prediction
    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0][1]

    st.divider()
    st.subheader("Prediction Result")

    col1, col2 = st.columns(2)

    with col1:
        if prediction == 1:
            st.error("⚠️ Customer is likely to Churn")
        else:
            st.success("✅ Customer is likely to Stay")

    with col2:
        st.metric(
            "Churn Probability",
            f"{probability * 100:.2f}%"
        )

    st.progress(float(probability))

    st.caption(
        "The probability is a model estimate, not a guarantee "
        of the customer's future behavior."
    )

### Run the Application
From a terminal in the project folder, run:

```bash
pip install streamlit pandas numpy scikit-learn joblib
streamlit run app.py
```

The interface accepts customer data, displays the churn prediction clearly, and reports the churn probability.


In [ ]:
!pip install streamlit pandas numpy scikit-learn joblib

In [ ]:
!pip install pyngrok

In [ ]:
import subprocess
import time

subprocess.Popen(["streamlit", "run", "app.py"])
time.sleep(3)

!ssh -o StrictHostKeyChecking=no -R 80:localhost:8501 serveo.net

# 11. Package and Download the Project

This section collects the trained model, Streamlit application, dependency file, and notebook into one ZIP file, then downloads it directly when the notebook is running in Google Colab.


In [ ]:
from pathlib import Path
import shutil
import sys

project_folder = Path('/content/Customer_Churn_Project')
project_folder.mkdir(parents=True, exist_ok=True)

# Save the dependency list used by the Streamlit application.
requirements = """streamlit
pandas
numpy
scikit-learn
joblib
"""
(project_folder / 'requirements.txt').write_text(requirements, encoding='utf-8')

# Copy generated project files when they exist.
files_to_copy = ['app.py', 'churn_model.pkl', 'model_features.pkl']
for file_name in files_to_copy:
    source = Path(file_name)
    if source.exists():
        shutil.copy2(source, project_folder / source.name)
        print(f'Added: {source.name}')
    else:
        print(f'Skipped because it was not found: {source.name}')

# Copy the current notebook from Colab when its file is available.
notebook_candidates = list(Path('/content').glob('*.ipynb'))
if notebook_candidates:
    current_notebook = max(notebook_candidates, key=lambda p: p.stat().st_mtime)
    shutil.copy2(current_notebook, project_folder / 'Customer_Churn_End_to_End_Project.ipynb')
    print(f'Added notebook: {current_notebook.name}')
else:
    print('Notebook file was not found automatically. Download/save the notebook separately if needed.')

# Create the ZIP archive.
zip_path = shutil.make_archive(
    '/content/Customer_Churn_End_to_End_Project',
    'zip',
    root_dir=project_folder
)
print('Project ZIP created:', zip_path)


In [ ]:
# Download the ZIP automatically in Google Colab.
try:
    from google.colab import files
    files.download('/content/Customer_Churn_End_to_End_Project.zip')
except ImportError:
    print('Automatic download is available in Google Colab only.')
    print('ZIP location:', '/content/Customer_Churn_End_to_End_Project.zip')


> Run the packaging cell after training and saving the final model, then run the download cell. Your browser will download `Customer_Churn_End_to_End_Project.zip`.
